# Análise Havas - Engajamento Neural vs Performance

Este notebook aplica os mesmos testes realizados nas bases GAIA e Tunad:
1. Análise de Correlação Linear (Pearson)
2. Análise de Comparação de Grupos (Quartis)
3. Regressão Linear Conjunta e XGBoost
4. Análise de Multicolinearidade
5. Análise Temporal (5 segundos para vídeos)
6. Análises Específicas por Segmento

In [ ]:
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
from scipy.stats import pearsonr, spearmanr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Try to import XGBoost, but continue if not available
try:
    from xgboost import XGBRegressor
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost not available. Skipping XGBoost analysis.")

## 1. Carregar Dados

In [ ]:
# Carregar índices de IA dos arquivos JSON
creatives_base = 'CSA_BETC Havas_Forebrain/creatives'
json_files = glob(os.path.join(creatives_base, '**', '*.json'), recursive=True)

ai_data = []

for json_file in json_files:
    try:
        with open(json_file, 'r') as f:
            data = json.load(f)
        
        # Extrair nome do arquivo criativo (remover .json)
        creative_path = json_file[:-5]  # Remove .json
        creative_name = os.path.basename(creative_path)
        
        # Determinar se é imagem ou vídeo
        is_video = creative_name.lower().endswith(('.mp4', '.mov', '.avi'))
        
        row = {
            'creative_path': creative_path,
            'creative_name': creative_name,
            'is_video': is_video,
            'json_path': json_file
        }
        
        # Para imagens: valores únicos
        # Para vídeos: arrays (calcular média e outras métricas)
        if is_video:
            if 'engajamentoneural' in data and isinstance(data['engajamentoneural'], list):
                neural_array = np.array(data['engajamentoneural'])
                row['engajamento_neural_mean'] = float(np.mean(neural_array))
                row['engajamento_neural_std'] = float(np.std(neural_array))
                row['engajamento_neural_max'] = float(np.max(neural_array))
                row['engajamento_neural_min'] = float(np.min(neural_array))
                
                # Primeiros 5 segundos (assumindo ~30 fps, ~150 frames)
                if 'tempos' in data and isinstance(data['tempos'], list):
                    tempos = np.array(data['tempos'])
                    mask_5s = tempos <= 5.0
                    if np.any(mask_5s):
                        neural_5s = neural_array[mask_5s]
                        row['engajamento_neural_5s'] = float(np.mean(neural_5s))
                    else:
                        # Fallback: primeiros 150 frames
                        neural_5s = neural_array[:min(150, len(neural_array))]
                        row['engajamento_neural_5s'] = float(np.mean(neural_5s))
                else:
                    # Fallback: primeiros 150 frames
                    neural_5s = neural_array[:min(150, len(neural_array))]
                    row['engajamento_neural_5s'] = float(np.mean(neural_5s))
            else:
                row['engajamento_neural_mean'] = data.get('engajamentoneural', np.nan)
                row['engajamento_neural_5s'] = np.nan
            
            if 'focus' in data and isinstance(data['focus'], list):
                focus_array = np.array(data['focus'])
                row['foco_mean'] = float(np.mean(focus_array))
                row['foco_std'] = float(np.std(focus_array))
            else:
                row['foco_mean'] = data.get('focus', np.nan)
            
            if 'cognitivedemand' in data and isinstance(data['cognitivedemand'], list):
                demand_array = np.array(data['cognitivedemand'])
                row['demanda_cognitiva_mean'] = float(np.mean(demand_array))
                row['demanda_cognitiva_std'] = float(np.std(demand_array))
            else:
                row['demanda_cognitiva_mean'] = data.get('cognitivedemand', np.nan)
        else:
            # Imagem: valores únicos
            row['engajamento_neural_mean'] = data.get('engajamentoneural', np.nan)
            row['foco_mean'] = data.get('focus', np.nan)
            row['demanda_cognitiva_mean'] = data.get('cognitivedemand', np.nan)
            row['engajamento_neural_5s'] = np.nan  # Não aplicável para imagens
        
        ai_data.append(row)
    except Exception as e:
        print(f"Erro ao processar {json_file}: {e}")
        continue

df_ai = pd.DataFrame(ai_data)
print(f"Carregados {len(df_ai)} criativos com índices de IA")
print(f"Imagens: {len(df_ai[~df_ai['is_video']])}")
print(f"Vídeos: {len(df_ai[df_ai['is_video']])}")
df_ai.head()

In [ ]:
# Carregar dados de performance
df_perf = pd.read_csv('CSA_BETC Havas_Forebrain/sourceadsdata.csv')

# Limpar e converter colunas numéricas
def clean_currency(value):
    if pd.isna(value) or value == '#DIV/0!':
        return np.nan
    if isinstance(value, str):
        # Remove R$, espaços e vírgulas
        value = value.replace('R$', '').replace(' ', '').replace(',', '.')
    try:
        return float(value)
    except:
        return np.nan

def clean_percentage(value):
    if pd.isna(value) or value == '#DIV/0!':
        return np.nan
    if isinstance(value, str):
        value = value.replace('%', '').replace(',', '.')
    try:
        return float(value)
    except:
        return np.nan

# Converter colunas
df_perf['Investimento'] = df_perf['Investimento'].apply(clean_currency)
df_perf['Impressões'] = pd.to_numeric(df_perf['Impressões'], errors='coerce')
df_perf['Cliques'] = pd.to_numeric(df_perf['Cliques'], errors='coerce')
df_perf['Pedidos'] = pd.to_numeric(df_perf['Pedidos'], errors='coerce')
df_perf['CPM'] = df_perf['CPM'].apply(clean_currency)
df_perf['CPC'] = df_perf['CPC'].apply(clean_currency)
df_perf['TxClickCTRImp'] = df_perf['TxClickCTRImp'].apply(clean_percentage)

# Calcular CTR se não existir ou estiver vazio
df_perf['CTR'] = df_perf['TxClickCTRImp']
df_perf.loc[df_perf['CTR'].isna(), 'CTR'] = (df_perf['Cliques'] / df_perf['Impressões'] * 100)

# Filtrar apenas linhas não excluídas
df_perf = df_perf[df_perf['Exclude'] != True].copy()

print(f"Carregados {len(df_perf)} registros de performance")
df_perf.head()

In [ ]:
# Tentar fazer matching entre criativos e performance
# Estratégia: usar informações do caminho do arquivo e do CSV

def extract_keywords_from_path(path):
    """Extrai palavras-chave do caminho do arquivo"""
    path_lower = path.lower()
    keywords = []
    
    # Extrair formato
    if 'estatic' in path_lower:
        keywords.append('estatico')
    elif 'anim' in path_lower:
        keywords.append('animado')
    elif 'carrossel' in path_lower:
        keywords.append('carrossel')
    
    # Extrair criativo (kv-hero, pais-custo-beneficio, etc)
    if 'kv-hero' in path_lower:
        keywords.append('kv-hero')
    elif 'custo-beneficio' in path_lower or 'custo beneficio' in path_lower:
        keywords.append('custo-beneficio')
    elif 'gb-preco' in path_lower or 'gb preco' in path_lower:
        keywords.append('gb-preco')
    elif 'chip-gratis' in path_lower or 'chip gratis' in path_lower:
        keywords.append('chip-gratis')
    
    return keywords

def extract_keywords_from_csv_row(row):
    """Extrai palavras-chave da linha do CSV"""
    keywords = []
    
    formato = str(row.get('Formato', '')).lower()
    if formato:
        keywords.append(formato)
    
    criativo = str(row.get('Criativo', '')).lower()
    if criativo:
        keywords.append(criativo)
    
    return keywords

# Adicionar keywords aos dataframes
df_ai['keywords'] = df_ai['creative_path'].apply(extract_keywords_from_path)
df_perf['keywords'] = df_perf.apply(extract_keywords_from_csv_row, axis=1)

# Tentar matching simples (pode ser refinado depois)
# Por enquanto, vamos trabalhar com os dados que temos
# e fazer análises separadas quando necessário

print("Matching básico concluído. Nota: Matching perfeito pode requerer refinamento manual.")

## 2. Análise de Correlação Linear (Pearson)

In [ ]:
# Para esta análise, vamos trabalhar com os dados de IA disponíveis
# e criar métricas agregadas quando possível

# Preparar dados para análise
df_analysis = df_ai.copy()

# Se tivermos matching, podemos fazer análises mais detalhadas
# Por enquanto, vamos focar nas análises possíveis com os dados de IA

print("=== Análise de Correlação Interna dos Índices ===")
print("\nCorrelação entre os índices de IA:")

indices_cols = ['engajamento_neural_mean', 'foco_mean', 'demanda_cognitiva_mean']
available_cols = [col for col in indices_cols if col in df_analysis.columns]

if len(available_cols) >= 2:
    corr_matrix = df_analysis[available_cols].corr(method='pearson')
    print(corr_matrix)
    
    # Correlações específicas
    if 'engajamento_neural_mean' in available_cols and 'foco_mean' in available_cols:
        corr_neural_foco = pearsonr(
            df_analysis['engajamento_neural_mean'].dropna(),
            df_analysis['foco_mean'].dropna()
        )
        print(f"\nEngajamento Neural vs Foco: r={corr_neural_foco[0]:.3f}, p={corr_neural_foco[1]:.4f}")
    
    if 'engajamento_neural_mean' in available_cols and 'demanda_cognitiva_mean' in available_cols:
        corr_neural_demand = pearsonr(
            df_analysis['engajamento_neural_mean'].dropna(),
            df_analysis['demanda_cognitiva_mean'].dropna()
        )
        print(f"Engajamento Neural vs Demanda Cognitiva: r={corr_neural_demand[0]:.3f}, p={corr_neural_demand[1]:.4f}")
    
    if 'foco_mean' in available_cols and 'demanda_cognitiva_mean' in available_cols:
        corr_foco_demand = pearsonr(
            df_analysis['foco_mean'].dropna(),
            df_analysis['demanda_cognitiva_mean'].dropna()
        )
        print(f"Foco vs Demanda Cognitiva: r={corr_foco_demand[0]:.3f}, p={corr_foco_demand[1]:.4f}")

## 3. Análise de Comparação de Grupos (Quartis)

In [ ]:
# Análise de quartis (top 25% vs bottom 25%)
# Nota: Para análise completa, precisaríamos de métricas de performance
# Por enquanto, vamos mostrar a distribuição dos índices

def quartile_analysis(df, index_col, performance_col=None):
    """Análise de quartis para um índice"""
    if index_col not in df.columns:
        print(f"Coluna {index_col} não encontrada")
        return
    
    # Remover NaN
    df_clean = df[[index_col]].dropna()
    if performance_col:
        df_clean = df[[index_col, performance_col]].dropna()
    
    if len(df_clean) < 4:
        print(f"Dados insuficientes para análise de quartis ({len(df_clean)} registros)")
        return
    
    # Calcular quartis
    q25 = df_clean[index_col].quantile(0.25)
    q75 = df_clean[index_col].quantile(0.75)
    
    bottom_25 = df_clean[df_clean[index_col] <= q25]
    top_25 = df_clean[df_clean[index_col] >= q75]
    
    print(f"\n=== Análise de Quartis: {index_col} ===")
    print(f"Bottom 25%: {len(bottom_25)} registros (<= {q25:.2f})")
    print(f"Top 25%: {len(top_25)} registros (>= {q75:.2f})")
    
    if performance_col and performance_col in df_clean.columns:
        perf_bottom = bottom_25[performance_col].mean()
        perf_top = top_25[performance_col].mean()
        diff = perf_top - perf_bottom
        print(f"\nPerformance média (Bottom 25%): {perf_bottom:.2f}")
        print(f"Performance média (Top 25%): {perf_top:.2f}")
        print(f"Diferença: {diff:.2f} ({diff/perf_bottom*100:.1f}%)")
    else:
        print(f"\nÍndice médio (Bottom 25%): {bottom_25[index_col].mean():.2f}")
        print(f"Índice médio (Top 25%): {top_25[index_col].mean():.2f}")

# Análise para cada índice
for idx_col in ['engajamento_neural_mean', 'foco_mean', 'demanda_cognitiva_mean']:
    if idx_col in df_analysis.columns:
        quartile_analysis(df_analysis, idx_col)

## 4. Regressão Linear Conjunta

In [ ]:
# Regressão linear conjunta
# Nota: Para regressão completa, precisaríamos de variável target (performance)
# Vamos preparar o código para quando tivermos matching

def regression_analysis(X, y, feature_names):
    """Análise de regressão linear"""
    if len(X) < len(feature_names) + 1:
        print(f"Dados insuficientes para regressão ({len(X)} amostras, {len(feature_names)} features)")
        return None
    
    # Remover NaN
    mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
    X_clean = X[mask]
    y_clean = y[mask]
    
    if len(X_clean) < len(feature_names) + 1:
        print(f"Dados insuficientes após limpeza ({len(X_clean)} amostras)")
        return None
    
    # Regressão linear
    model = LinearRegression()
    model.fit(X_clean, y_clean)
    
    y_pred = model.predict(X_clean)
    r2 = r2_score(y_clean, y_pred)
    mse = mean_squared_error(y_clean, y_pred)
    
    print(f"\n=== Regressão Linear ===")
    print(f"R²: {r2:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"\nCoeficientes:")
    for name, coef in zip(feature_names, model.coef_):
        print(f"  {name}: {coef:.4f}")
    print(f"  Intercepto: {model.intercept_:.4f}")
    
    return model

print("Função de regressão preparada. Execute quando tiver dados de performance matched.")

## 5. Modelagem XGBoost

In [ ]:
if HAS_XGBOOST:
    def xgboost_analysis(X, y, feature_names):
        """Análise com XGBoost"""
        if len(X) < len(feature_names) + 1:
            print(f"Dados insuficientes para XGBoost ({len(X)} amostras)")
            return None
        
        # Remover NaN
        mask = ~(np.isnan(X).any(axis=1) | np.isnan(y))
        X_clean = X[mask]
        y_clean = y[mask]
        
        if len(X_clean) < 10:
            print(f"Dados insuficientes após limpeza ({len(X_clean)} amostras)")
            return None
        
        # Split train/test
        X_train, X_test, y_train, y_test = train_test_split(
            X_clean, y_clean, test_size=0.2, random_state=42
        )
        
        # XGBoost
        model = XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
        model.fit(X_train, y_train)
        
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)
        
        r2_train = r2_score(y_train, y_pred_train)
        r2_test = r2_score(y_test, y_pred_test)
        
        print(f"\n=== XGBoost Regressor ===")
        print(f"R² (Train): {r2_train:.4f}")
        print(f"R² (Test): {r2_test:.4f}")
        
        # Feature importance
        importance = model.feature_importances_
        print(f"\nFeature Importance:")
        for name, imp in zip(feature_names, importance):
            print(f"  {name}: {imp:.4f}")
        
        return model
    
    print("Função XGBoost preparada. Execute quando tiver dados de performance matched.")
else:
    print("XGBoost não disponível. Instale com: pip install xgboost")

## 6. Análise Temporal (Primeiros 5 Segundos)

In [ ]:
# Análise temporal para vídeos (primeiros 5 segundos)
df_videos = df_analysis[df_analysis['is_video']].copy()

if len(df_videos) > 0:
    print(f"=== Análise Temporal (Vídeos) ===")
    print(f"Total de vídeos: {len(df_videos)}")
    
    # Comparar média total vs primeiros 5 segundos
    if 'engajamento_neural_mean' in df_videos.columns and 'engajamento_neural_5s' in df_videos.columns:
        df_videos_clean = df_videos[['engajamento_neural_mean', 'engajamento_neural_5s']].dropna()
        
        if len(df_videos_clean) > 0:
            print(f"\nVídeos com dados temporais: {len(df_videos_clean)}")
            print(f"\nEngajamento Neural (média total): {df_videos_clean['engajamento_neural_mean'].mean():.2f}")
            print(f"Engajamento Neural (primeiros 5s): {df_videos_clean['engajamento_neural_5s'].mean():.2f}")
            
            # Correlação entre média total e primeiros 5s
            corr_temporal = pearsonr(
                df_videos_clean['engajamento_neural_mean'],
                df_videos_clean['engajamento_neural_5s']
            )
            print(f"\nCorrelação (média total vs 5s): r={corr_temporal[0]:.3f}, p={corr_temporal[1]:.4f}")
        else:
            print("Dados temporais insuficientes")
    else:
        print("Colunas de análise temporal não encontradas")
else:
    print("Nenhum vídeo encontrado para análise temporal")

## 7. Análises por Segmento

In [ ]:
# Análises por segmento (Formato, Tipo de criativo, etc.)

def analyze_by_segment(df, segment_col, index_cols):
    """Análise por segmento"""
    if segment_col not in df.columns:
        print(f"Coluna de segmento '{segment_col}' não encontrada")
        return
    
    segments = df[segment_col].dropna().unique()
    print(f"\n=== Análise por Segmento: {segment_col} ===")
    print(f"Segmentos encontrados: {len(segments)}")
    
    for segment in segments:
        df_seg = df[df[segment_col] == segment]
        print(f"\n--- {segment} ({len(df_seg)} registros) ---")
        
        for idx_col in index_cols:
            if idx_col in df_seg.columns:
                values = df_seg[idx_col].dropna()
                if len(values) > 0:
                    print(f"  {idx_col}: média={values.mean():.2f}, std={values.std():.2f}")

# Análise por tipo (imagem vs vídeo)
if 'is_video' in df_analysis.columns:
    print("\n=== Comparação: Imagens vs Vídeos ===")
    
    for media_type in [False, True]:
        type_name = "Vídeos" if media_type else "Imagens"
        df_type = df_analysis[df_analysis['is_video'] == media_type]
        
        print(f"\n{type_name} ({len(df_type)} registros):")
        for idx_col in ['engajamento_neural_mean', 'foco_mean', 'demanda_cognitiva_mean']:
            if idx_col in df_type.columns:
                values = df_type[idx_col].dropna()
                if len(values) > 0:
                    print(f"  {idx_col}: média={values.mean():.2f}, std={values.std():.2f}")

## 8. Resumo e Próximos Passos

In [ ]:
print("=== RESUMO DA ANÁLISE ===")
print(f"\nTotal de criativos processados: {len(df_ai)}")
print(f"  - Imagens: {len(df_ai[~df_ai['is_video']])}")
print(f"  - Vídeos: {len(df_ai[df_ai['is_video']])}")
print(f"\nTotal de registros de performance: {len(df_perf)}")

print("\n=== PRÓXIMOS PASSOS ===")
print("1. Refinar matching entre criativos e dados de performance")
print("2. Executar análises de correlação com métricas de performance")
print("3. Executar regressões com variáveis de performance como target")
print("4. Análises segmentadas por campanha, formato, criativo")
print("5. Comparar resultados com análises GAIA e Tunad")